In [ ]:
! pip install -q sentence-transformers

In [ ]:
!pip install -q diffusers accelerate torch

In [ ]:
import json
import os
import numpy as np
from tqdm import tqdm
import torch

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from diffusers import StableDiffusionPipeline

from diffusers import DPMSolverMultistepScheduler

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
PROJECT_ROOT = (
    "/content/drive/MyDrive/Colab Notebooks/"
    "ProfessionAI_AIengineering/9. Generative AI/"
    "Project_Generative_AI"
)

DATA_DIR = os.path.join(PROJECT_ROOT, "data")
TEXT_VARIATIONS_DIR = os.path.join(DATA_DIR, "text_variations")

TEXT_VARIATION_FILE = os.path.join(TEXT_VARIATIONS_DIR, "text_variations_train_small.json")

# Load JSON File

In [ ]:
with open(TEXT_VARIATION_FILE, "r") as f:
    text_variations = json.load(f)

OVERVIEW OF WHAT WE’LL DO

For each image:

Load original + generated captions

Compute sentence embeddings

Score each generated caption using:

Semantic similarity to original captions

Diversity from other generated captions

Select top 2

Save new JSON → selected_captions_train_small.json

In [ ]:
text_variations

# Load Sentence Embedding Model

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

# Define Caption Scoring Function

We want high similarity to original captions and low similarity to other generated captions.

In [ ]:
def select_top_captions(original_captions, generated_captions, top_k=2):

    if len(generated_captions) <= top_k:
        return generated_captions

    # Encode captions
    orig_embeddings = model.encode(original_captions)
    gen_embeddings = model.encode(generated_captions)

    scores = []

    for i, gen_emb in enumerate(gen_embeddings):

        # Similarity to original captions
        sim_to_orig = cosine_similarity(
            [gen_emb], orig_embeddings
        ).mean()

        # Similarity to other generated captions (diversity)
        other_indices = [j for j in range(len(gen_embeddings)) if j != i]

        if other_indices:
            sim_to_gen = cosine_similarity(
                [gen_emb],
                gen_embeddings[other_indices]
            ).mean()
        else:
            sim_to_gen = 0

        # Final weighted score
        score = 0.7 * sim_to_orig - 0.3 * sim_to_gen
        scores.append(score)

    # Select top_k captions
    top_indices = np.argsort(scores)[-top_k:]
    selected = [generated_captions[i] for i in top_indices]

    return selected


# Apply selection to all samples

In [ ]:
selected_data = {}

for idx, data in tqdm(text_variations.items()):

    class_name = data["class_name"]
    original_captions = data["original_captions"]
    generated_captions = data["generated_captions"]

    selected_generated = select_top_captions(
        original_captions,
        generated_captions,
        top_k=2
    )

    selected_data[idx] = {
        "class_name": class_name,
        "original_captions": original_captions,
        "selected_generated_captions": selected_generated
    }

In [ ]:
selected_data

# Save new JSON File

In [ ]:
SELECTED_FILE = os.path.join(
    TEXT_VARIATIONS_DIR,
    "selected_captions_train_small.json"
)

with open(SELECTED_FILE, "w") as f:
    json.dump(selected_data, f, indent=4)

Sentence embeddings (all-MiniLM-L6-v2) were used to compute semantic similarity between generated captions and original captions. Captions were ranked using a weighted scoring function balancing semantic alignment and inter-caption diversity. The top two captions per image were selected.

In [ ]:
# SELECTED_FILE = os.path.join(
#     TEXT_VARIATIONS_DIR,
#     "selected_captions_train_small.json"
# )

# with open(SELECTED_FILE, "r") as f:
#     selected_data = json.load(f)

In [ ]:
# output directory for synthetic images
SYNTHETIC_DIR = os.path.join(DATA_DIR, "synthetic", "images")
os.makedirs(SYNTHETIC_DIR, exist_ok=True)

# Load Stable Diffusion Model

In [ ]:

device = "cuda" if torch.cuda.is_available() else "cpu"

pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
)


pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to(device)

pipe.enable_attention_slicing()
pipe.safety_checker = None
torch.backends.cuda.matmul.allow_tf32 = True

In [ ]:
# selected_data

In [ ]:
def build_prompt(caption, class_name):

    prompt = (
        f"A high-resolution realistic photograph of a {class_name}, "
        f"{caption.lower()}"
    )

    return prompt

In [ ]:
CHECKPOINT_DIR = os.path.join(DATA_DIR, "synthetic")

In [ ]:
CHECKPOINT_FILE = os.path.join(CHECKPOINT_DIR, "generation_metadata_checkpoint.json")

if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE, "r") as f:
        generation_metadata = json.load(f)
    print("Checkpoint loaded.")
else:
    generation_metadata = {}

In [ ]:
# batched generation loop

BATCH_SIZE = 4

for idx, data in tqdm(selected_data.items()):

    class_name = data["class_name"]
    captions = data["selected_generated_captions"]

    if idx not in generation_metadata:
      generation_metadata[idx] = []

    # Skip already processed captions
    already_done = len(generation_metadata[idx])
    captions = captions[already_done:]


    # batch loop
    for batch_start in range(0, len(captions), BATCH_SIZE):

        batch_captions = captions[batch_start:batch_start + BATCH_SIZE]

        prompts = [
            build_prompt(caption, class_name)
            for caption in batch_captions
        ]

        images = pipe(
            prompts,
            num_inference_steps=20,
            guidance_scale=7.5,
            height=512,
            width=512
        ).images

        # Save each image from batch
        for i, image in enumerate(images):

            global_index = already_done + batch_start + i
            image_filename = f"{idx}_{global_index}.png"
            image_path = os.path.join(SYNTHETIC_DIR, image_filename)

            image.save(image_path)

            generation_metadata[idx].append({
                "image_path": image_path,
                "class_name": class_name,
                "prompt": prompts[i],
                "source": "synthetic"
            })

        #SAVE CHECKPOINT AFTER EACH BATCH
        with open(CHECKPOINT_FILE, "w") as f:
            json.dump(generation_metadata, f, indent=4)

        torch.cuda.empty_cache()

In [ ]:
FINAL_METADATA_FILE = os.path.join(CHECKPOINT_DIR, "generation_metadata_final.json")

with open(FINAL_METADATA_FILE, "w") as f:
    json.dump(generation_metadata, f, indent=4)

print("Final metadata saved.")

In [ ]:
CHECKPOINT_DIR

In [ ]:
SYNTHETIC_DIR

In [ ]:
num_images = len(os.listdir(SYNTHETIC_DIR))
print("Total generated images:", num_images)

In [ ]:
import random
from PIL import Image
import matplotlib.pyplot as plt

sample_images = random.sample(os.listdir(SYNTHETIC_DIR), 5)

for img_name in sample_images:
    img = Image.open(os.path.join(SYNTHETIC_DIR, img_name))
    plt.figure()
    plt.imshow(img)
    plt.title(img_name)
    plt.axis("off")